In [1]:
import csv
import json
import re
import warnings
from pathlib import Path

import torch
import torch._dynamo  # pre-initialize before transformers to avoid circular import
import spacy

RAW_DIR = Path("../data/raw/daicwoz")
OUT_FILE = Path("../data/processed/daicwoz_finetune.jsonl")
OUT_FILE.parent.mkdir(parents=True, exist_ok=True)

EXCLUDE_SESSIONS = {451, 458, 480}

SYSTEM_PROMPT = """\
You are Ellie, a warm and attentive virtual clinical interviewer conducting \
a structured mental health screening. Your goal is to assess the person's \
current mood, sleep, energy, concentration, social engagement, and any \
trauma or distress they may be experiencing. Ask one question at a time, \
listen carefully, and follow up naturally on what they share. Be gentle \
and non-judgmental. If the person hints at self-harm or suicidal thoughts, \
respond with direct, compassionate concern."""

NOISE_PATTERN = re.compile(r"<[^>]+>|(?<!\w)xxx(?!\w)", re.IGNORECASE)
TAG_PAREN_PATTERN = re.compile(r"^\w+\s+\((.+)\)$")
SENTENCE_START_PATTERN = re.compile(r"([.!?]\s+)([a-z])")
PRONOUN_I_PATTERN = re.compile(r"\bi\b")
ELLIE_PATTERN = re.compile(r"\bellie\b", re.IGNORECASE)
# Matches underscore-separated single letters: l_a -> LA, p_t_s_d -> PTSD
ABBREV_PATTERN = re.compile(r"\b([a-z])(_[a-z])+\b")

def expand_abbreviations(text: str) -> str:
    return ABBREV_PATTERN.sub(lambda m: m.group(0).replace("_", "").upper(), text)

def clean(text: str) -> str:
    m = TAG_PAREN_PATTERN.match(text.strip())
    if m:
        text = m.group(1)
    text = NOISE_PATTERN.sub("", text)
    text = expand_abbreviations(text)
    return " ".join(text.split()).strip()

def capitalize_sentences(text: str) -> str:
    text = text[:1].upper() + text[1:] if text else text
    return SENTENCE_START_PATTERN.sub(lambda m: m.group(1) + m.group(2).upper(), text)

def fix_pronoun_i(text: str) -> str:
    return PRONOUN_I_PATTERN.sub("I", text)

def fix_ellie(text: str) -> str:
    return ELLIE_PATTERN.sub("Ellie", text)

def capitalize_proper_nouns(text: str, nlp) -> str:
    doc = nlp(text)
    return "".join(
        (token.text if token.text.isupper() else token.text.capitalize()) + token.whitespace_
        if token.pos_ == "PROPN"
        else token.text_with_ws
        for token in doc
    )

def load_conversation(csv_path: Path) -> list[dict]:
    rows = []
    with open(csv_path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f, delimiter="\t")
        for row in reader:
            speaker = row["speaker"].strip()
            value = clean(row["value"])
            if not value:
                continue
            role = "assistant" if speaker == "Ellie" else "user"
            if rows and rows[-1]["role"] == role:
                rows[-1]["content"] += " " + value
            else:
                rows.append({"role": role, "content": value})
    return rows

# --- Load all conversations ---
print("Loading conversations...")
all_conversations = []


Loading conversations...


## ESConv Preprocessing

Convert ESConv emotional support conversations to the same chat JSONL format.
`supporter` → `assistant` (Ellie), `seeker` → `user`.

In [2]:
import json
import warnings
from pathlib import Path

ESC_RAW_DIR = Path("../data/raw/esconv")
ESC_OUT_FILE = Path("../data/processed/esconv_finetune.jsonl")
ESC_OUT_FILE.parent.mkdir(parents=True, exist_ok=True)

SYSTEM_PROMPT = """\
You are a caring and empathetic supporter helping someone work through \
emotional distress. Listen actively, acknowledge their feelings, and help \
them feel heard and understood. You can offer gentle reframing, \
encouragement, or practical suggestions when appropriate, but prioritize \
validation and emotional support over advice. Be warm, patient, and \
non-judgmental throughout the conversation."""

def load_esconv_conversations(raw_dir: Path) -> list[list[dict]]:
    conversations = []
    for split_file in sorted(raw_dir.glob("*.jsonl")):
        with split_file.open(encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                outer = json.loads(line)
                # Record is stored as a JSON string inside the "text" field
                inner = json.loads(outer["text"]) if isinstance(outer.get("text"), str) else outer
                dialog = inner.get("dialog", [])
                turns = []
                for utt in dialog:
                    speaker = utt.get("speaker", "")
                    text = utt.get("text", "").strip()
                    if not text:
                        continue
                    # sys = supporter (assistant/therapist), usr = seeker (user)
                    role = "assistant" if speaker == "sys" else "user"
                    if turns and turns[-1]["role"] == role:
                        turns[-1]["content"] += " " + text
                    else:
                        turns.append({"role": role, "content": text})
                if len(turns) >= 2:
                    conversations.append(turns)
    return conversations

print("Loading ESConv conversations...")
esconv_conversations = load_esconv_conversations(ESC_RAW_DIR)
print(f"Loaded {len(esconv_conversations)} conversations")
print(f"\nExample (first 2 turns):")
for msg in esconv_conversations[0][:2]:
    print(f"  [{msg['role']}]: {msg['content'][:120]!r}")


Loading ESConv conversations...
Loaded 1300 conversations

Example (first 2 turns):
  [assistant]: 'Hello. How are you today?'
  [user]: 'hi i am okay, a little bit sad though'


In [4]:
from dotenv import load_dotenv
load_dotenv("../.env")

import warnings
import re
import torch
import torch._dynamo  # pre-initialize before transformers to avoid circular import
import spacy
from transformers.pipelines import TokenClassificationPipeline
from deepmultilingualpunctuation import PunctuationModel

# Patch grouped_entities compatibility (safe to apply multiple times)
if not getattr(TokenClassificationPipeline, "_grouped_entities_patched", False):
    _orig_sanitize = TokenClassificationPipeline._sanitize_parameters

    def _patched_sanitize(self, **kwargs):
        if "grouped_entities" in kwargs:
            kwargs["aggregation_strategy"] = "simple" if kwargs.pop("grouped_entities") else "none"
        return _orig_sanitize(self, **kwargs)

    TokenClassificationPipeline._sanitize_parameters = _patched_sanitize
    TokenClassificationPipeline._grouped_entities_patched = True

SENTENCE_START_PATTERN = re.compile(r"([.!?]\s+)([a-z])")
PRONOUN_I_PATTERN = re.compile(r"\bi\b")
ELLIE_PATTERN = re.compile(r"\bellie\b", re.IGNORECASE)

def capitalize_sentences(text):
    text = text[:1].upper() + text[1:] if text else text
    return SENTENCE_START_PATTERN.sub(lambda m: m.group(1) + m.group(2).upper(), text)

def fix_pronoun_i(text):
    return PRONOUN_I_PATTERN.sub("I", text)

def fix_ellie(text):
    return ELLIE_PATTERN.sub("Ellie", text)

def capitalize_proper_nouns(text, nlp):
    doc = nlp(text)
    return "".join(
        (token.text if token.text.isupper() else token.text.capitalize()) + token.whitespace_
        if token.pos_ == "PROPN"
        else token.text_with_ws
        for token in doc
    )

print("Loading punctuation model...")
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    punct_model = PunctuationModel()

print("Loading spaCy model...")
nlp = spacy.load("en_core_web_sm")

coords, texts = [], []
for c_idx, turns in enumerate(esconv_conversations):
    for t_idx, turn in enumerate(turns):
        coords.append((c_idx, t_idx))
        texts.append(turn["content"])

print(f"Processing {len(texts)} turns with punctuation model...")
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    restored_texts = [punct_model.restore_punctuation(t) for t in texts]

print("Applying capitalization fixes...")
for (c_idx, t_idx), restored in zip(coords, restored_texts):
    text = capitalize_sentences(restored)
    text = fix_pronoun_i(text)
    text = capitalize_proper_nouns(text, nlp)
    text = fix_ellie(text)
    esconv_conversations[c_idx][t_idx]["content"] = text
print("Done.")

records_written = 0
with ESC_OUT_FILE.open("w", encoding="utf-8") as out_f:
    for turns in esconv_conversations:
        messages = [{"role": "system", "content": SYSTEM_PROMPT}] + turns
        out_f.write(json.dumps({"messages": messages}, ensure_ascii=False) + "\n")
        records_written += 1

print(f"Wrote {records_written} conversations → {ESC_OUT_FILE}")

Loading punctuation model...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading spaCy model...
Processing 30392 turns with punctuation model...


[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Applying capitalization fixes...
Done.
Wrote 1300 conversations → ../data/processed/esconv_finetune.jsonl


## AnnoMI Preprocessing

Convert AnnoMI motivational-interviewing transcripts to chat JSONL format.  
`therapist` → `assistant`, `client` → `user`.  
Only high-quality MI conversations are included by default (set `MI_QUALITY_FILTER = None` to include all).


In [5]:
import csv
import json
from pathlib import Path

ANNOMI_RAW_FILE = Path("../data/raw/annomi/dataset.csv")
ANNOMI_OUT_FILE = Path("../data/processed/annomi_finetune.jsonl")
ANNOMI_OUT_FILE.parent.mkdir(parents=True, exist_ok=True)

# Set to "high", "low", or None (include both)
MI_QUALITY_FILTER = "high"

SYSTEM_PROMPT = """\
You are a therapist using Motivational Interviewing (MI) to help someone \
explore their ambivalence about making a positive change. Use open-ended \
questions, affirmations, reflective listening, and summaries (OARS) to \
elicit and strengthen the person's own motivation for change. Avoid \
arguing, persuading, or pushing — follow the person's lead, roll with \
resistance, and highlight their strengths and values."""

def load_annomi_conversations(csv_path: Path, mi_quality: str | None = "high") -> list[list[dict]]:
    # Group rows by transcript_id, preserving utterance_id order
    transcripts: dict[int, list[dict]] = {}
    with csv_path.open(newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            if mi_quality and row["mi_quality"].strip().lower() != mi_quality.lower():
                continue
            tid = int(row["transcript_id"])
            transcripts.setdefault(tid, []).append(row)

    conversations = []
    for tid in sorted(transcripts):
        rows = sorted(transcripts[tid], key=lambda r: int(r["utterance_id"]))
        turns: list[dict] = []
        for row in rows:
            text = row["utterance_text"].strip()
            if not text:
                continue
            role = "assistant" if row["interlocutor"].strip().lower() == "therapist" else "user"
            if turns and turns[-1]["role"] == role:
                turns[-1]["content"] += " " + text
            else:
                turns.append({"role": role, "content": text})
        if len(turns) >= 2:
            conversations.append(turns)
    return conversations

print(f"Loading AnnoMI conversations (mi_quality={MI_QUALITY_FILTER!r})...")
annomi_conversations = load_annomi_conversations(ANNOMI_RAW_FILE, mi_quality=MI_QUALITY_FILTER)
print(f"Loaded {len(annomi_conversations)} conversations")
print(f"\nExample (first 2 turns):")
for msg in annomi_conversations[0][:2]:
    print(f"  [{msg['role']}]: {msg['content'][:120]!r}")


Loading AnnoMI conversations (mi_quality='high')...
Loaded 110 conversations

Example (first 2 turns):
  [assistant]: 'Thanks for filling it out. We give this form to everyone once a year regardless of why they come in. It helps us provide'
  [user]: 'Sure.'


In [6]:
import json

records_written = 0
with ANNOMI_OUT_FILE.open("w", encoding="utf-8") as out_f:
    for turns in annomi_conversations:
        messages = [{"role": "system", "content": SYSTEM_PROMPT}] + turns
        out_f.write(json.dumps({"messages": messages}, ensure_ascii=False) + "\n")
        records_written += 1

print(f"Wrote {records_written} conversations → {ANNOMI_OUT_FILE}")

Wrote 110 conversations → ../data/processed/annomi_finetune.jsonl


## Merge Datasets (optional)

Combine DAIC-WOZ and ESConv into a single JSONL for joint fine-tuning.

In [3]:
import random
from pathlib import Path

MERGED_OUT = Path("../data/processed/combined_finetune_nodaicwoz.jsonl")
TRAIN_OUT  = Path("../data/processed/combined_train_nodaicwoz.jsonl")
VAL_OUT    = Path("../data/processed/combined_val_nodaicwoz.jsonl")

daicwoz_lines = Path("../data/processed/daicwoz_finetune.jsonl").read_text(encoding="utf-8").splitlines()
esconv_lines  = Path("../data/processed/esconv_finetune.jsonl").read_text(encoding="utf-8").splitlines()
annomi_lines  = Path("../data/processed/annomi_finetune.jsonl").read_text(encoding="utf-8").splitlines()

all_lines = [l for l in esconv_lines + annomi_lines if l.strip()]

# Save full combined file (unchanged)
with MERGED_OUT.open("w", encoding="utf-8") as f:
    f.write("\n".join(all_lines) + "\n")

# ── 90/10 train/val split ──────────────────────────────────────────────────
random.seed(42)
indices = list(range(len(all_lines)))
random.shuffle(indices)
split_idx = int(len(indices) * 0.9)
train_idx, val_idx = indices[:split_idx], indices[split_idx:]

train_lines = [all_lines[i] for i in train_idx]
val_lines   = [all_lines[i] for i in val_idx]

with TRAIN_OUT.open("w", encoding="utf-8") as f:
    f.write("\n".join(train_lines) + "\n")
with VAL_OUT.open("w", encoding="utf-8") as f:
    f.write("\n".join(val_lines) + "\n")

print(f"DAIC-WOZ : {len(daicwoz_lines)} conversations")
print(f"ESConv   : {len(esconv_lines)} conversations")
print(f"AnnoMI   : {len(annomi_lines)} conversations")
print(f"Combined : {len(all_lines)} conversations → {MERGED_OUT}")
print(f"Train    : {len(train_lines)} conversations → {TRAIN_OUT}")
print(f"Val      : {len(val_lines)} conversations → {VAL_OUT}")

DAIC-WOZ : 186 conversations
ESConv   : 1300 conversations
AnnoMI   : 110 conversations
Combined : 1410 conversations → ../data/processed/combined_finetune_nodaicwoz.jsonl
Train    : 1269 conversations → ../data/processed/combined_train_nodaicwoz.jsonl
Val      : 141 conversations → ../data/processed/combined_val_nodaicwoz.jsonl
